# Vergleich faulty vs clean DB

Einfaches Einmal-Notebook zur Artefakt-Erstellung.

Erzeugt:
- Monatliche Vergleichstabelle als CSV
- Zwei Vergleichsplots als PNG

Voraussetzung: Beide DB-Dateien existieren bereits.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Pfade bei Bedarf anpassen
FAULTY_DB_PATH = Path('../data_output/dwh_data_faulty.db')
CLEAN_DB_PATH = Path('../data_output/dwh_data.db')

ARTIFACT_DIR = Path('../data_output/plots')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not FAULTY_DB_PATH.exists():
    raise FileNotFoundError(f'Faulty DB fehlt: {FAULTY_DB_PATH.resolve()}')
if not CLEAN_DB_PATH.exists():
    raise FileNotFoundError(f'Clean DB fehlt: {CLEAN_DB_PATH.resolve()}')

print('DBs gefunden:')
print('faulty:', FAULTY_DB_PATH.resolve())
print('clean :', CLEAN_DB_PATH.resolve())
print('Artefakte in:', ARTIFACT_DIR.resolve())

In [ ]:
def monthly_counts_from_db(db_path: Path, source_label: str) -> pd.DataFrame:
    query = '''
    WITH monthly_newspapers AS (
        SELECT
            substr(data_published, 1, 7) AS month,
            COUNT(*) AS newspapers_count
        FROM newspapers
        WHERE data_published IS NOT NULL
        GROUP BY substr(data_published, 1, 7)
    ),
    monthly_context AS (
        SELECT
            substr(n.data_published, 1, 7) AS month,
            COUNT(*) AS context_count
        FROM context c
        JOIN newspapers n ON n.newspaper_id = c.newspaper_id
        WHERE n.data_published IS NOT NULL
        GROUP BY substr(n.data_published, 1, 7)
    ),
    all_months AS (
        SELECT month FROM monthly_newspapers
        UNION
        SELECT month FROM monthly_context
    )
    SELECT
        m.month,
        COALESCE(n.newspapers_count, 0) AS newspapers_count,
        COALESCE(c.context_count, 0) AS context_count
    FROM all_months m
    LEFT JOIN monthly_newspapers n ON n.month = m.month
    LEFT JOIN monthly_context c ON c.month = m.month
    ORDER BY m.month
    '''

    with sqlite3.connect(str(db_path)) as conn:
        df = pd.read_sql_query(query, conn)

    df['source'] = source_label
    df['month_dt'] = pd.to_datetime(df['month'] + '-01', errors='coerce')
    return df

monthly = pd.concat([
    monthly_counts_from_db(FAULTY_DB_PATH, 'faulty'),
    monthly_counts_from_db(CLEAN_DB_PATH, 'clean')
], ignore_index=True)

monthly = monthly.sort_values(['month', 'source']).reset_index(drop=True)
display(monthly.head(20))

In [ ]:
csv_path = ARTIFACT_DIR / 'db_monthly_comparison_faulty_vs_clean.csv'
monthly.to_csv(csv_path, index=False)

pivot_newspapers = monthly.pivot(index='month_dt', columns='source', values='newspapers_count').sort_index()
pivot_context = monthly.pivot(index='month_dt', columns='source', values='context_count').sort_index()

# Plot styles: different linestyles for faulty vs clean, semi-transparent lines, transparent saved PNGs
styles = {'faulty': '--', 'clean': ':'}
marker_size = 6
line_alpha = 0.8

fig1, ax1 = plt.subplots(figsize=(12, 4.8))
for col in pivot_newspapers.columns:
    linestyle = styles.get(col, '-')
    ax1.plot(pivot_newspapers.index, pivot_newspapers[col], marker='o', markersize=marker_size, linewidth=2,
             linestyle=linestyle, alpha=line_alpha, label=col)
ax1.axvspan(pd.Timestamp('2025-02-01'), pd.Timestamp('2025-03-01'), alpha=0.08, color='red')
ax1.set_title('newspapers pro Monat: faulty vs clean')
ax1.set_xlabel('Monat')
ax1.set_ylabel('Anzahl')
ax1.grid(alpha=0.3)
ax1.legend()
fig1.tight_layout()
# Save with transparent background
fig1.patch.set_alpha(0)
plot1_path = ARTIFACT_DIR / 'db_compare_newspapers_monthly_faulty_vs_clean.png'
fig1.savefig(plot1_path, dpi=140, bbox_inches='tight', transparent=True)
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 4.8))
for col in pivot_context.columns:
    linestyle = styles.get(col, '-')
    ax2.plot(pivot_context.index, pivot_context[col], marker='o', markersize=marker_size, linewidth=2,
             linestyle=linestyle, alpha=line_alpha, label=col)
ax2.axvspan(pd.Timestamp('2025-02-01'), pd.Timestamp('2025-03-01'), alpha=0.08, color='red')
ax2.set_title('context pro Monat: faulty vs clean')
ax2.set_xlabel('Monat')
ax2.set_ylabel('Anzahl')
ax2.grid(alpha=0.3)
ax2.legend()
fig2.tight_layout()
fig2.patch.set_alpha(0)
plot2_path = ARTIFACT_DIR / 'db_compare_context_monthly_faulty_vs_clean.png'
fig2.savefig(plot2_path, dpi=140, bbox_inches='tight', transparent=True)
plt.show()

print('Artefakte geschrieben:')
print('-', csv_path.resolve())
print('-', plot1_path.resolve())
print('-', plot2_path.resolve())

In [ ]:
feb = monthly[monthly['month'].str.endswith('-02')].copy()
display(feb.sort_values(['month', 'source']).reset_index(drop=True))